# MuseTalk - Coach IA
GPU T4 requis

In [ ]:
# CELLULE 1 - Installation
!pip install edge-tts -q
!git clone https://github.com/TMElyralab/MuseTalk.git
%cd MuseTalk
!pip install -r requirements.txt -q
print('Installation OK')

In [ ]:
# CELLULE 2 - Modeles
from huggingface_hub import snapshot_download
snapshot_download(repo_id='TMElyralab/MuseTalk', local_dir='./models/musetalk')
print('Modeles OK')

In [ ]:
# CELLULE 3 - Upload image
from google.colab import files
import shutil, os
os.makedirs('./data/images', exist_ok=True)
uploaded = files.upload()
for fn in uploaded:
    shutil.move(fn, f'./data/images/{fn}')
    IMAGE_PATH = f'./data/images/{fn}'
    print(f'Image OK: {IMAGE_PATH}')

In [ ]:
# CELLULE 4 - Generer audios
import edge_tts, asyncio, os
os.makedirs('./data/audio', exist_ok=True)

SCRIPTS = {
    '01-google-symptomes': 'J ai google mes symptomes. Fatigue apres chaque repas. Ventre qui gonfle. Envie de sucre a 16h. Google m a dit prediabete. Mon medecin aussi. Mais au lieu de paniquer, j ai juste change ce que je mange. Pas un regime. Un vrai plan adapte a mon corps. 10 semaines plus tard. Plus de coups de barre. 8 kilos en moins. Et mon medecin qui comprend pas. Commente GLYCEMIE si tu veux savoir ce que j ai change.',
    '02-medecin-choque': 'Mon medecin m a dit de prendre des medicaments. Pour mon cholesterol. A 42 ans. J ai dit non. Donnez-moi 10 semaines. J ai pas fait de sport. J ai pas compte les calories. J ai juste mange les bons aliments. Au bon moment. 10 semaines plus tard, mon bilan sanguin etait parfait. Mon medecin m a demande ce que j avais fait. Je lui ai montre mon assiette. Commente ton age, je te dis par quoi commencer.',
    '03-arrete-petit-dej': 'J ai arrete le petit-dejeuner classique. Pain, confiture, jus d orange. Mon nutritionniste m a explique pourquoi j avais faim a 10h. Pic de glycemie. Puis crash. Puis fringale. Depuis 10 semaines je mange autrement le matin. Plus jamais faim avant midi. J ai perdu 6 kilos sans effort. Commente MATIN si tu veux savoir ce que je mange maintenant.',
    '04-balance-mentait': 'Ma balance me mentait. 85 kilos depuis 3 ans. Je mangeais moins. Je bougeais plus. Rien ne changeait. Puis j ai compris. C etait pas les calories. C etait l inflammation. En 10 semaines j ai change ce que je mange. Pas combien. Moins 9 kilos. Et pour la premiere fois, c est reste. Commente ton poids, je t explique pourquoi ca bloque.',
    '05-ventre-gonfle': 'Ventre gonfle apres chaque repas. Ballonnements. Fatigue. Je pensais que c etait normal. Mon medecin aussi. Puis j ai decouvert que 3 aliments que je mangeais tous les jours causaient tout ca. En 10 semaines j ai tout change. Ventre plat. Energie stable. Digestion parfaite. Commente VENTRE si tu veux la liste des 3 aliments.',
    '06-triglycerides': 'Triglycerides trop hauts. Mon medecin voulait me mettre sous statines. J ai demande 3 mois. Il m a donne 2. En 8 semaines mes triglycerides ont baisse de 40 pourcent. Sans medicaments. Juste en changeant mon alimentation. Mon medecin n en revenait pas. Commente BILAN si tu veux mon plan.',
    '07-regime-yoyo': 'J ai fait tous les regimes. Weight Watchers. Keto. Jeune intermittent. Je perdais. Je reprenais. A chaque fois plus. Puis j ai compris que le probleme c etait pas ma volonte. C etait mon metabolisme. En 10 semaines j ai repare ca. Moins 11 kilos. Et ca fait 6 mois que c est stable. Commente YOYO si t en as marre de reprendre.',
    '08-glycemie-cachee': 'Ta glycemie te tue en silence. Tu manges bien. Tu fais du sport. Mais t es fatigue. Tu stockes du gras. Tu comprends pas. J etais pareil. Puis j ai mesure ma glycemie apres les repas. Le choc. Des pics a 180. Avec des aliments soi-disant sains. 10 semaines de changements. Glycemie stable. 7 kilos en moins. Commente GLYCEMIE pour comprendre.',
    '09-tour-de-taille': 'Mon tour de taille disait plus que ma balance. 102 centimetres. Zone rouge. Risque cardio. Risque diabete. En 10 semaines je suis passe a 89. Pas avec du cardio. Pas avec un regime. Juste en mangeant les bons aliments aux bons moments. Commente ta taille en centimetres, je te dis ou t en es.',
    '10-apres-50-ans': 'Apres 50 ans perdre du poids c est pas pareil. Le metabolisme ralentit. Les hormones changent. Les regimes classiques marchent plus. J ai trouve une methode adaptee a mon age. En 10 semaines j ai perdu 8 kilos. Sans me priver. Sans forcer. En mangeant mieux pas moins. Commente ton age je t envoie le plan adapte.'
}

async def gen():
    for name, text in SCRIPTS.items():
        c = edge_tts.Communicate(text, 'fr-FR-HenriNeural')
        await c.save(f'./data/audio/{name}.mp3')
        print(f'OK: {name}')

await gen()
print('Audios OK!')

In [ ]:
# CELLULE 5 - Generer videos
import subprocess, glob
os.makedirs('./results', exist_ok=True)
audio_files = sorted(glob.glob('./data/audio/*.mp3'))
for i, audio in enumerate(audio_files, 1):
    name = os.path.splitext(os.path.basename(audio))[0]
    print(f'[{i}/{len(audio_files)}] {name}...')
    r = subprocess.run(['python', '-m', 'scripts.inference', '--driven_audio', audio, '--source_image', IMAGE_PATH, '--result_dir', './results'], capture_output=True, text=True)
    print('  OK' if r.returncode == 0 else f'  ERREUR: {r.stderr[-300:]}')
print('Termine!')

In [ ]:
# CELLULE 6 - Telecharger
from google.colab import files
!cd results && zip -r /content/tiktok_videos.zip *.mp4
files.download('/content/tiktok_videos.zip')